# 11-00_XGBoost_Regressor
Das Notebook dokumentiert die methodische Pipeline zur Entwicklung, Optimierung und Evaluierung eines Machine-Learning-Modells auf Basis des XGBoost-Algorithmus.

Die Implementierung gliedert sich in die folgenden prozeduralen Schritte:

* **Initialisierung und Setup:** Zu Beginn erfolgt die Einrichtung der Arbeitsumgebung sowie der Import der erforderlichen Softwarebibliotheken, welche für die algorithmische Modellierung und die grafische Auswertung (u. a. mittels Plotly) notwendig sind.
* **Hyperparameter-Optimierung:** Um die Generalisierungsfähigkeit des Modells zu maximieren, wird eine systematische Rastersuche (Grid Search) durchgeführt. Diese dient der Identifikation der optimalen Modellparameter, wie beispielsweise der maximalen Baumtiefe (`max_depth`).
* **Modellkalibrierung:** Das finale Training des XGBoost-Modells wird unter Verwendung der durch die Rastersuche empirisch ermittelten Bestparameter vollzogen.
* **Post-Processing und Rangbildung:** Die kontinuierlichen Prädiktionswerte des trainierten Modells werden in einem nachgelagerten Schritt transformiert, um eine diskrete, ordinale Rangfolge (Ranking) zu generieren.
* **Evaluation und Validierung:** Abschließend erfolgt eine Überprüfung der Modellgüte durch den Vergleich der prädizierten Ergebnisse mit einer definierten Baseline. Die Validierung wird auf Ebene der einzelnen Gruppierungsinstanzen (z. B. anhand eines `sample_race`) durchgeführt, um die Validität der berechneten Rankings systematisch zu prüfen.

Importe und Daten laden

In [1]:
import os
os.chdir('/Applications/GADA/GADA-Group3-Cycling-Stage-Prediction/src/Notebooks')

import pandas as pd
import numpy as np
import xgboost as xgb
import os
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.model_selection import ParameterGrid
from scipy.stats import spearmanr
from sklearn.metrics import ndcg_score

# --- 1. Daten laden ---
pfad = "../../data/processed/"
X_train = pd.read_pickle(os.path.join(pfad, "X_train.pkl"))
y_class_train = pd.read_pickle(os.path.join(pfad, "y_class_train.pkl"))
X_val = pd.read_pickle(os.path.join(pfad, "X_val.pkl"))
y_class_val = pd.read_pickle(os.path.join(pfad, "y_class_val.pkl"))
meta_val = pd.read_pickle(os.path.join(pfad, "meta_val.pkl"))
y_rank_val = pd.read_pickle(os.path.join(pfad, "y_rank_val.pkl"))

y_train = y_class_train['target_ordinal']
y_val = y_class_val['target_ordinal']

# Gewichte berechnen (Imbalance-Korrektur)
print("Berechne Klassengewichte für das Training...")
gewichte_train = compute_sample_weight(class_weight='balanced', y=y_train)
gewichte_val = compute_sample_weight(class_weight='balanced', y=y_val)

Berechne Klassengewichte für das Training...


Visualisierung: Korrektur des Klassen-Ungleichgewichts im XGBoost Regressor

In [2]:
import plotly.graph_objects as go
import plotly.subplots as sp

# 1. Echte Daten aus y_train und gewichte_train auslesen
unique_classes, counts = np.unique(y_train, return_counts=True)
total_rows = len(y_train)

# Wir lesen das echte, berechnete Gewicht für jede Klasse aus
weight_values = [gewichte_train[y_train == cls][0] for cls in unique_classes]

# Effektiver Einfluss = Echte Zeilenanzahl * Mathematisches Gewicht
effective_influence = counts * weight_values

df_summary = pd.DataFrame({
    'Klasse': ['Hauptfeld (0)', 'Top 20 (1)', 'Top 10 (2)', 'Top 5 (3)'],
    'Echte_Anzahl': counts,
    'Effektiver_Einfluss': effective_influence
})

# Reihenfolge für den Plot anpassen: Top 5 nach links
df_summary = df_summary.set_index('Klasse').loc[['Top 5 (3)', 'Top 10 (2)', 'Top 20 (1)', 'Hauptfeld (0)']].reset_index()

colors = ['#d95f52', '#d5813b', '#d8b32e', '#3c4c5e']

# 2. Zwei Diagramme erstellen (Vorher vs. Nachher)
fig = sp.make_subplots(rows=1, cols=2, 
                       subplot_titles=("Vorher: Echte Zeilenverteilung<br>(Starkes Ungleichgewicht)", 
                                       "Nachher: Effektiver Lerneinfluss<br>(Durch sample_weight balanciert)"),
                       horizontal_spacing=0.15)

# Plot 1: Unkorrigierte Verteilung
for i, row in df_summary.iterrows():
    fig.add_trace(go.Bar(
        x=[row['Klasse']], 
        y=[row['Echte_Anzahl']],
        marker_color=colors[i],
        text=[f"{(row['Echte_Anzahl'] / total_rows * 100):.1f}%<br>({row['Echte_Anzahl']:,.0f})"],
        textposition='outside',
        showlegend=False
    ), row=1, col=1)

# Plot 2: Effektiver Einfluss beim Lernen
for i, row in df_summary.iterrows():
    fig.add_trace(go.Bar(
        x=[row['Klasse']], 
        y=[row['Effektiver_Einfluss']],
        marker_color=colors[i],
        # Wir zeigen den neuen fiktiven Zeilenwert im Text
        text=[f"{row['Effektiver_Einfluss']:,.0f}"],
        textposition='outside',
        showlegend=False
    ), row=1, col=2)

# 3. Layout anpassen
fig.update_layout(
    title={"text": "Korrektur des Klassen-Ungleichgewichts im XGBoost Regressor<br><span style='font-size: 14px; font-weight: normal;'>Vergleich der physischen Zeilen mit dem ausbalancierten Einfluss auf die Verlustfunktion</span>"},
    height=600,
    width=1000,
    plot_bgcolor='white',
    font=dict(size=14),
    margin=dict(t=140)  # NEU: Mehr Platz oben für die Titel!
)

fig.update_yaxes(title_text="Anzahl der Fahrer-Zeilen", row=1, col=1, showgrid=True, gridcolor='lightgray', rangemode='tozero')
fig.update_yaxes(title_text="Gesamteinfluss (Zeilen × Gewicht)", row=1, col=2, showgrid=True, gridcolor='lightgray', rangemode='tozero')

for i in range(1, 3):
    fig.update_xaxes(showline=True, linewidth=1, linecolor='lightgray', row=1, col=i)
    fig.update_yaxes(showline=True, linewidth=1, linecolor='lightgray', row=1, col=i)
    
# NEU: Die y-Achsen nach oben hin etwas strecken, damit der Text über den Balken nicht abgeschnitten wird
fig.update_yaxes(range=[0, df_summary['Echte_Anzahl'].max() * 1.15], row=1, col=1)
fig.update_yaxes(range=[0, df_summary['Effektiver_Einfluss'].max() * 1.15], row=1, col=2)

# Diagramm direkt im Notebook anzeigen
fig.show()

Grid Search (Hyperparameter-Tuning)

In [3]:
# --- 2. Grid Search ---
param_grid = {
    'max_depth': [5, 7],               
    'learning_rate': [0.01, 0.05],     
    'subsample': [0.8],                
    'colsample_bytree': [0.8],         
    'reg_alpha': [0, 0.5, 1.0],        
    'reg_lambda': [1.0, 5.0]           
}

grid = list(ParameterGrid(param_grid))
print(f"Starte Grid Search mit {len(grid)} verschiedenen Kombinationen...\n")

best_score = float('inf') 
best_params = None

for i, params in enumerate(grid):
    model = xgb.XGBRegressor(
        objective='reg:squarederror',
        n_estimators=1500,          
        early_stopping_rounds=50,   
        random_state=42,
        enable_categorical=True,
        **params
    )
    
    model.fit(
        X_train, y_train,
        sample_weight=gewichte_train,
        eval_set=[(X_val, y_val)],
        sample_weight_eval_set=[gewichte_val],
        verbose=False
    )
    
    if model.best_score < best_score:
        best_score = model.best_score
        best_params = params
        
    print(f"Lauf {i+1} beendet. RMSE: {model.best_score:.5f}")

print("\nBeste Parameter:", best_params)

Starte Grid Search mit 24 verschiedenen Kombinationen...

Lauf 1 beendet. RMSE: 1.01428
Lauf 2 beendet. RMSE: 1.01522
Lauf 3 beendet. RMSE: 1.01442
Lauf 4 beendet. RMSE: 1.01595
Lauf 5 beendet. RMSE: 1.01568
Lauf 6 beendet. RMSE: 1.01423
Lauf 7 beendet. RMSE: 1.01641
Lauf 8 beendet. RMSE: 1.01759
Lauf 9 beendet. RMSE: 1.01731
Lauf 10 beendet. RMSE: 1.01756
Lauf 11 beendet. RMSE: 1.01761
Lauf 12 beendet. RMSE: 1.01740
Lauf 13 beendet. RMSE: 1.01605
Lauf 14 beendet. RMSE: 1.01308
Lauf 15 beendet. RMSE: 1.01626
Lauf 16 beendet. RMSE: 1.01540
Lauf 17 beendet. RMSE: 1.01396
Lauf 18 beendet. RMSE: 1.01615
Lauf 19 beendet. RMSE: 1.01827
Lauf 20 beendet. RMSE: 1.01480
Lauf 21 beendet. RMSE: 1.01618
Lauf 22 beendet. RMSE: 1.01545
Lauf 23 beendet. RMSE: 1.01626
Lauf 24 beendet. RMSE: 1.01580

Beste Parameter: {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 5, 'reg_alpha': 0, 'reg_lambda': 5.0, 'subsample': 0.8}


Das finale Modell mit den besten Parametern

In [4]:
# --- 3. Finales Training mit den besten Parametern ---
xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=1500,
    random_state=42,
    early_stopping_rounds=50,
    enable_categorical=True,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0,
    reg_lambda=5.0
)

print("Starte finales Modelltraining...")
xgb_model.fit(
    X_train, y_train,
    sample_weight=gewichte_train, 
    eval_set=[(X_train, y_train), (X_val, y_val)],
    sample_weight_eval_set=[gewichte_train, gewichte_val], 
    verbose=50 
)

Starte finales Modelltraining...
[0]	validation_0-rmse:1.10940	validation_1-rmse:1.11093
[50]	validation_0-rmse:0.99069	validation_1-rmse:1.02749
[100]	validation_0-rmse:0.96948	validation_1-rmse:1.01808
[150]	validation_0-rmse:0.95711	validation_1-rmse:1.01564
[200]	validation_0-rmse:0.94589	validation_1-rmse:1.01375
[250]	validation_0-rmse:0.93515	validation_1-rmse:1.01412
[271]	validation_0-rmse:0.93108	validation_1-rmse:1.01397


,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,50
,enable_categorical,True
,eval_metric,None


Post-Processing (Ranking erstellen)

In [5]:
# --- 4. Post-Processing & Ranking generieren ---
preds_continuous = xgb_model.predict(X_val)

df_results = meta_val.copy().reset_index(drop=True)
true_ranks_clean = y_rank_val['rank'].reset_index(drop=True)

df_results['raw_prediction'] = preds_continuous
df_results['true_rank'] = true_ranks_clean

# Nach Rennen gruppieren und sortieren
df_results = df_results.sort_values(by=['stage_id', 'raw_prediction'], ascending=[True, False])
df_results['predicted_rank'] = df_results.groupby('stage_id')['raw_prediction'].rank(method='first', ascending=False)

In [6]:

# --- 5. Physikalisch korrekte Klassen erzwingen ---
conditions = [
    (df_results['predicted_rank'] <= 5),
    (df_results['predicted_rank'] <= 10),
    (df_results['predicted_rank'] <= 20)
]
choices = [3, 2, 1]
df_results['forced_ordinal_class'] = np.select(conditions, choices, default=0)


# --- 6. Ergebnis anschauen ---
print("\nFertig! Hier ist ein Beispiel-Rennen aus den Validierungsdaten:")
sample_race = df_results['stage_id'].iloc[0]
display_df = df_results[df_results['stage_id'] == sample_race]

print(display_df[['meta_name', 'true_rank', 'predicted_rank', 'raw_prediction', 'forced_ordinal_class']].head(25))


# --- 7. Ergebnis als Pickle-Datei speichern ---
output_file = "../../data/models/xgboost_regressor_results.pkl"

output_dir = os.path.dirname(output_file)
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

df_results.to_pickle(output_file)
print(f"\nDie Ergebnisse wurden erfolgreich unter '{output_file}' gespeichert.")


Fertig! Hier ist ein Beispiel-Rennen aus den Validierungsdaten:
                meta_name  true_rank  predicted_rank  raw_prediction  \
4613     Michael Matthews      141.0             1.0        2.153720   
4478        Mads Pedersen        4.0             2.0        2.131749   
4612         Kaden Groves      140.0             3.0        1.993400   
4495       Geraint Thomas       22.0             4.0        1.924386   
4603      Alberto Dainese      131.0             5.0        1.916959   
4501      Thymen Arensman       28.0             6.0        1.906357   
4485         Lorenzo Rota       11.0             7.0        1.902703   
4554    Vincenzo Albanese       81.0             8.0        1.901552   
4497         João Almeida       24.0             9.0        1.866239   
4518   Andreas Leknessund       45.0            10.0        1.842879   
4523      Simone Consonni       50.0            11.0        1.838693   
4492      Alberto Bettiol       19.0            12.0        1.836195   

Evaluierung (Spearman & NDCG)

In [7]:
# --- 5. Evaluierung der Baseline ---
df_eval = df_results.copy()

# A: Spearman
spearman_scores = []
for stage, group in df_eval.groupby('stage_id'):
    if len(group) > 1:
        corr, _ = spearmanr(group['true_rank'], group['predicted_rank'])
        if not np.isnan(corr):
            spearman_scores.append(corr)

# B: NDCG
conditions_true = [
    (df_eval['true_rank'] <= 5),
    (df_eval['true_rank'] <= 10),
    (df_eval['true_rank'] <= 20)
]
df_eval['true_relevance'] = np.select(conditions_true, [3, 2, 1], default=0)

ndcg_scores_5 = []
ndcg_scores_10 = []
ndcg_scores_20 = []

for stage, group in df_eval.groupby('stage_id'):
    if group['true_relevance'].sum() > 0:
        y_true = group['true_relevance'].values.reshape(1, -1)
        y_score = group['raw_prediction'].values.reshape(1, -1)
        
        ndcg_scores_5.append(ndcg_score(y_true, y_score, k=5))
        ndcg_scores_10.append(ndcg_score(y_true, y_score, k=10))
        ndcg_scores_20.append(ndcg_score(y_true, y_score, k=20))

print(f"Durchschnittliche Spearman-Korrelation: {np.mean(spearman_scores):.4f}")
print(f"Durchschnittlicher NDCG@5: {np.mean(ndcg_scores_5):.4f}")
print(f"Durchschnittlicher NDCG@10: {np.mean(ndcg_scores_10):.4f}")
print(f"Durchschnittlicher NDCG@20: {np.mean(ndcg_scores_20):.4f}")

Durchschnittliche Spearman-Korrelation: 0.3260
Durchschnittlicher NDCG@5: 0.4516
Durchschnittlicher NDCG@10: 0.4307
Durchschnittlicher NDCG@20: 0.4413


In [8]:
sample_race = df_eval['stage_id'].iloc[0]
check_df = df_eval[df_eval['stage_id'] == sample_race]
print(check_df[['stage_id','meta_name', 'true_rank', 'predicted_rank', 'raw_prediction']].head(20))

                     stage_id           meta_name  true_rank  predicted_rank  \
4613  giro-d-italia_2023_ST10    Michael Matthews      141.0             1.0   
4478  giro-d-italia_2023_ST10       Mads Pedersen        4.0             2.0   
4612  giro-d-italia_2023_ST10        Kaden Groves      140.0             3.0   
4495  giro-d-italia_2023_ST10      Geraint Thomas       22.0             4.0   
4603  giro-d-italia_2023_ST10     Alberto Dainese      131.0             5.0   
4501  giro-d-italia_2023_ST10     Thymen Arensman       28.0             6.0   
4485  giro-d-italia_2023_ST10        Lorenzo Rota       11.0             7.0   
4554  giro-d-italia_2023_ST10   Vincenzo Albanese       81.0             8.0   
4497  giro-d-italia_2023_ST10        João Almeida       24.0             9.0   
4518  giro-d-italia_2023_ST10  Andreas Leknessund       45.0            10.0   
4523  giro-d-italia_2023_ST10     Simone Consonni       50.0            11.0   
4492  giro-d-italia_2023_ST10     Albert